# Imports

In [78]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report


In [76]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Loading data and exploring it 

In [24]:
train_df =pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-testing-master.csv")

# Handle Churn column properly (fill NaN and convert to int)
train_df['Churn'] = train_df['Churn'].fillna(0).astype(int)
test_df['Churn'] = test_df['Churn'].astype(int)

In [25]:
# Training data
print("First 5 rows of the training data:")
train_df.head()

First 5 rows of the training data:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1


In [26]:
train_df.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440833.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567106
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [27]:
train_df['Churn'].value_counts()
train_df.isnull().sum()
train_df.info



<bound method DataFrame.info of         CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0              2.0  30.0  Female    39.0             14.0            5.0   
1              3.0  65.0  Female    49.0              1.0           10.0   
2              4.0  55.0  Female    14.0              4.0            6.0   
3              5.0  58.0    Male    38.0             21.0            7.0   
4              6.0  23.0    Male    32.0             20.0            5.0   
...            ...   ...     ...     ...              ...            ...   
440828    449995.0  42.0    Male    54.0             15.0            1.0   
440829    449996.0  25.0  Female     8.0             13.0            1.0   
440830    449997.0  26.0    Male    35.0             27.0            1.0   
440831    449998.0  28.0    Male    55.0             14.0            2.0   
440832    449999.0  31.0    Male    48.0             20.0            1.0   

        Payment Delay Subscription Type Contract Length

In [28]:
train_df.isnull().sum()
train_df.dtypes

CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                  int64
dtype: object

In [29]:
# Check unique values in 'Churn' before any changes
print("Unique values in 'Churn' before processing:")
print(train_df['Churn'].unique())
print("Value counts:")
print(train_df['Churn'].value_counts())

Unique values in 'Churn' before processing:
[1 0]
Value counts:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Data and feature Engineering 

In [30]:
# Handling missing values for numerical columns by filling them with the mean of each column
train_df.fillna(train_df.median(numeric_only= True), inplace=True)
test_df.fillna(train_df.median(numeric_only= True), inplace=True)

#filling missing values for categorical columns with "Na"
train_df.fillna("Na", inplace=True)
test_df.fillna("Na", inplace=True)


In [31]:
# Identify categorical columns (exclude Churn)
categorical_cols = train_df.select_dtypes(include=['object']).columns
if 'Churn' in categorical_cols:
    categorical_cols = categorical_cols.drop('Churn')

# Combine train and test for fitting encoder to ensure all categories are known
combined_df = pd.concat([train_df[categorical_cols], test_df[categorical_cols]], axis=0)

# Fit OneHotEncoder on combined data
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe.fit(combined_df)

# Transform train data
encoded_train = ohe.transform(train_df[categorical_cols])
encoded_train_df = pd.DataFrame(encoded_train, columns=ohe.get_feature_names_out(categorical_cols), index=train_df.index)
train_df = train_df.drop(categorical_cols, axis=1)
train_df = pd.concat([train_df, encoded_train_df], axis=1)

# Transform test data
encoded_test = ohe.transform(test_df[categorical_cols])
encoded_test_df = pd.DataFrame(encoded_test, columns=ohe.get_feature_names_out(categorical_cols), index=test_df.index)
test_df = test_df.drop(categorical_cols, axis=1)
test_df = pd.concat([test_df, encoded_test_df], axis=1)

In [32]:
#Splitting features and target variable (drop CustomerID as it's not predictive)
x = train_df.drop(['Churn', 'CustomerID'], axis=1)
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

# Use SMOTE for oversampling to balance classes
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(x, y)

Class distribution in y:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Modeling 

In [100]:
x_train , x_val , y_train , y_val = train_test_split(x_res, y_res, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

# Train the Logistic Regression model
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(x_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [82]:
# Get top features from XGBoost
feature_importances = pd.Series(model.feature_importances_, index=x.columns)
top_features = feature_importances.nlargest(10)
selected_features = top_features.index.tolist()
print("Top 10 Feature Importances from XGBoost:")
print(top_features)
print("Selected features:", selected_features)

Top 10 Feature Importances from XGBoost:
Support Calls                0.254200
Total Spend                  0.243810
Contract Length_Monthly      0.190102
Payment Delay                0.126328
Age                          0.059659
Gender_Male                  0.055113
Last Interaction             0.051887
Tenure                       0.009721
Usage Frequency              0.009180
Contract Length_Quarterly    0.000000
dtype: float32
Selected features: ['Support Calls', 'Total Spend', 'Contract Length_Monthly', 'Payment Delay', 'Age', 'Gender_Male', 'Last Interaction', 'Tenure', 'Usage Frequency', 'Contract Length_Quarterly']


In [101]:
# Predict on the validation set
y_pred = model.predict(x_val_scaled)

# Evaluate the model
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       0.88      0.93      0.90     49962
           1       0.93      0.87      0.90     50038

    accuracy                           0.90    100000
   macro avg       0.90      0.90      0.90    100000
weighted avg       0.90      0.90      0.90    100000



In [103]:
# Prepare test data (drop target column and any prediction columns)
columns_to_drop = ['Churn', 'CustomerID']
if 'Predicted_Churn' in test_df.columns:
    columns_to_drop.append('Predicted_Churn')
x_test = test_df.drop(columns_to_drop, axis=1)
x_test = x_test[selected_features]  # Select only the important features
x_test_scaled = scaler.transform(x_test)

# Predict on the test set with probability scores
test_probabilities = model.predict_proba(x_test_scaled)
# Use threshold 0.5
threshold = 0.5
test_predictions = (test_probabilities[:, 1] >= threshold).astype(int)

# Output the feature labels
print("Features used for prediction:")
print(list(x.columns))

# Add predictions as a new column to test_df
test_df['Predicted_Churn'] = test_predictions

# Compare predictions with actual values
print("\nFirst 10 rows comparison:")
print(test_df[['Churn', 'Predicted_Churn']].head(10))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_df['Churn'], test_df['Predicted_Churn'])
print("\nConfusion Matrix:")
print("[[TN, FP]")
print(" [FN, TP]]")
print(cm)

# Detailed comparison for first 20 rows
print("\nDetailed comparison (first 20 rows):")
comparison_df = test_df[['Churn', 'Predicted_Churn']].copy()
comparison_df['Correct'] = comparison_df['Churn'] == comparison_df['Predicted_Churn']
print(comparison_df.head(20))

# Show first 20 rows of test_df with all columns
print("\nFirst 20 rows of test_df with predictions:")
print(test_df.head(20))

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"\nAccuracy on test set: {accuracy:.4f}")

# Additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(test_df['Churn'], test_df['Predicted_Churn'])
recall = recall_score(test_df['Churn'], test_df['Predicted_Churn'])
f1 = f1_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Output the predictions
print("\nTest Predictions:")
print(test_predictions)

# Summary of predictions
import pandas as pd
print("\nPrediction Summary:")
pred_counts = pd.Series(test_predictions).value_counts()
print(pred_counts)

# Show that it's not all 1s
total_predictions = len(test_predictions)
ones_count = pred_counts.get(1, 0)
zeros_count = pred_counts.get(0, 0)
print(f"\nOut of {total_predictions} predictions:")
print(f"- {ones_count} predicted as 1 (churn)")
print(f"- {zeros_count} predicted as 0 (no churn)")
print(f"- Percentage of 1s: {ones_count/total_predictions*100:.1f}%")
print(f"- Percentage of 0s: {zeros_count/total_predictions*100:.1f}%")

# Feature coefficients
print("\nFeature Coefficients:")
for feature, coef in zip(x.columns, model.coef_[0]):
    print(f"{feature}: {coef:.4f}")


Features used for prediction:
['Support Calls', 'Total Spend', 'Contract Length_Monthly', 'Payment Delay', 'Age', 'Gender_Male', 'Last Interaction', 'Tenure', 'Usage Frequency', 'Contract Length_Quarterly']

First 10 rows comparison:
   Churn  Predicted_Churn
0      1                1
1      0                1
2      0                1
3      0                1
4      0                1
5      0                1
6      1                1
7      0                1
8      0                1
9      0                0

Confusion Matrix:
[[TN, FP]
 [FN, TP]]
[[ 7234 26647]
 [  374 30119]]

Detailed comparison (first 20 rows):
    Churn  Predicted_Churn  Correct
0       1                1     True
1       0                1    False
2       0                1    False
3       0                1    False
4       0                1    False
5       0                1    False
6       1                1     True
7       0                1    False
8       0                1    False
9       0  

In [104]:
# Confusion Matrix for test data
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_df['Churn'], test_df['Predicted_Churn'])
print("Confusion Matrix:")
print(cm)
print("\nTrue Negatives (TN):", cm[0,0])
print("False Positives (FP):", cm[0,1])
print("False Negatives (FN):", cm[1,0])
print("True Positives (TP):", cm[1,1])

Confusion Matrix:
[[ 7234 26647]
 [  374 30119]]

True Negatives (TN): 7234
False Positives (FP): 26647
False Negatives (FN): 374
True Positives (TP): 30119


In [75]:
# Get top 10 important features
import pandas as pd
feature_importances = pd.Series(model.feature_importances_, index=x.columns)
top_features = feature_importances.nlargest(20)
print("Top 10 Feature Importances:")
print(top_features)

# Select only top features
selected_features = top_features.index.tolist()
print("Selected features:", selected_features)

Top 10 Feature Importances:
Support Calls                0.344509
Total Spend                  0.239069
Contract Length_Monthly      0.152764
Age                          0.119019
Payment Delay                0.110469
Gender_Male                  0.013414
Last Interaction             0.012254
Contract Length_Quarterly    0.008218
Tenure                       0.000155
Usage Frequency              0.000128
dtype: float64
Selected features: ['Support Calls', 'Total Spend', 'Contract Length_Monthly', 'Age', 'Payment Delay', 'Gender_Male', 'Last Interaction', 'Contract Length_Quarterly', 'Tenure', 'Usage Frequency']


In [83]:
# Redefine x with selected features
x = train_df[selected_features]
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

# Use SMOTE for oversampling to balance classes
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(x, y)

Class distribution in y:
Churn
1    249999
0    190834
Name: count, dtype: int64


In [51]:
# Check test data Churn distribution
print("Test data Churn distribution:")
print(test_df['Churn'].value_counts())

Test data Churn distribution:
Churn
0    33881
1    30493
Name: count, dtype: int64
